# Assignment 6: Identify a material model for silicone

Use the force data from a **4 mm punch** to select a material model and its parameters. Then check whether that model predicts the forces for a **6 mm punch** without refitting.

Submit this notebook with your code, outputs, and written answers. Run it on one MPI process using the **Python (Finite Elements)** kernel ([setup instructions](https://github.com/CEADpx/finite-element-methods/blob/main/environment/README.md)).

## The indentation test

The rigid punch is moved downward by a prescribed depth $\delta$. Contact is frictionless, and the silicone is bonded to a rigid base. In this model the punch stays in contact with the same square patch throughout the test.



## The block and its boundary

Solve for the displacement $\boldsymbol u$ of the **silicone block only**. The rigid punch and base are not part of the FE mesh.



Use reference coordinates $\boldsymbol X=(X_1,X_2,X_3)^T$, with the origin at the lower front-left corner and $X_2$ upward. The reference domain is $\Omega_0=(0,8)\times(0,4)\times(0,8)$ in mm. Let $b$ be the side length of the square punch: $b=4$ mm for fitting and $b=6$ mm for prediction.

The block has three boundary regions. The figure shows them for the 4 mm punch.

- **Bonded bottom, $\Gamma_b$:** the entire face at $X_2=0$.
- **Punch footprint, $\Gamma_p$:** the square of side $b$ centered at $(4,4,4)$ on the top face.
- **Free surfaces, $\Gamma_f$:** the four sides and the top face outside the footprint.

In reference coordinates, these regions are

$$
\begin{aligned}
\Gamma_b &= [0,8]\times\{0\}\times[0,8],\\
\Gamma_p &= [4-b/2,\,4+b/2]\times\{4\}\times[4-b/2,\,4+b/2],\\
\Gamma_f &= \partial\Omega_0\setminus(\Gamma_b\cup\Gamma_p).
\end{aligned}
$$

## Strong form

Let $\boldsymbol P$ be the first Piola–Kirchhoff stress, $\boldsymbol N$ the outward unit normal to the reference block, and $\boldsymbol t_0=\boldsymbol P\boldsymbol N$ the traction per unit reference area. Here $\operatorname{Div}$ is taken with respect to $\boldsymbol X$. There is no body force.

$$
\begin{aligned}
-\operatorname{Div}\boldsymbol P &= \boldsymbol 0 &&\text{in }\Omega_0,\\
\boldsymbol u &= \boldsymbol 0 &&\text{on }\Gamma_b,\\
\boldsymbol P\boldsymbol N &= \boldsymbol 0 &&\text{on }\Gamma_f.
\end{aligned}
$$

In Task 1(a), choose the condition that completes the strong form on $\Gamma_p$. Once the displacement at depth $\delta$ is known, the block's upward force on the punch is the opposite of the punch's vertical force on the block:

$$
N(\delta)=-\int_{\Gamma_p} t_{0,2}\,\mathrm dA,
$$

where $t_{0,2}$ is the vertical component of $\boldsymbol t_0$. The supplied solver computes this force from the finite element reactions.

## The two material models

The deformation gradient is $\boldsymbol F=\boldsymbol I+\operatorname{Grad}\boldsymbol u$, where $\operatorname{Grad}$ is taken with respect to $\boldsymbol X$. Let $J=\det\boldsymbol F>0$ and $Q=J^{-2/3}\operatorname{tr}(\boldsymbol F^T\boldsymbol F)-3$. For either stored energy $\Psi$ below, $\boldsymbol P=\partial\Psi/\partial\boldsymbol F$.

$$
\begin{aligned}
\Psi_{\mathrm{NH}}(\boldsymbol F;\mu)
  &=\frac{\mu}{2}Q+\frac{K}{2}(\ln J)^2,
  &&\text{neo-Hookean},\\
\Psi_{\mathrm{SS}}(\boldsymbol F;\mu,\beta)
  &=\frac{\mu}{2}Q+\mu\beta Q^2+\frac{K}{2}(\ln J)^2,
  &&\text{strain-stiffening}.
\end{aligned}
$$

Both energies are per unit reference volume. The bulk modulus is $K=2.50$ MPa, $\mu$ is the initial shear modulus, and $\beta$ is dimensionless. The strain-stiffening energy with $\beta=0$ gives the neo-Hookean energy, so both models can use the same implementation.

## Force data and model selection

| Depth $\delta$ (mm) | 4 mm punch: force (N) | 6 mm punch: force (N) |
|---:|---:|---:|
| 0.30 | 4.20 | 6.70 |
| 0.60 | 10.25 | 17.10 |
| 0.90 | 19.90 | 34.90 |
| 1.20 | 34.20 | 63.20 |

The data are synthetic. Use **only the 4 mm punch data** to evaluate the candidates below.

| Model | Candidates for $\mu$ (MPa) | Candidates for $\beta$ |
|:--|:--|:--|
| Neo-Hookean | 0.30, 0.40, 0.50, 0.60, 0.70, 0.80 | 0 |
| Strain-stiffening | 0.30, 0.40, 0.50 | 0.50, 1.00, 1.50 |

Test every combination in each row. For a candidate, let $N_i^{\rm FE}$ be the force computed from the displacement solution at depth $\delta_i$, and let $N_i^{\rm obs}$ be the corresponding force in the 4 mm column. Compute its root-mean-square relative error:

$$
E=\left[\frac14\sum_{i=1}^{4}
  \left(\frac{N_i^{\rm FE}-N_i^{\rm obs}}{N_i^{\rm obs}}\right)^2\right]^{1/2}.
$$

Select the candidate with the smallest $E$ across both models. Reserve the 6 mm data for checking this selected fit.

## 1. Boundary conditions and variational forms (20 marks)

**(a)** Complete the strong form by choosing the conditions on $\Gamma_p$. Explain how your choice accounts for the prescribed motion and frictionless contact. Here $u_j$ and $t_{0,j}$ are components in the $X_j$ direction. Answer before reading the supplied solver.

| Choice | Conditions on $\Gamma_p$ |
|:--|:--|
| A | Prescribe a downward traction: $t_{0,2}=-p$, with $p>0$ given, and $t_{0,1}=t_{0,3}=0$. |
| B | Prescribe $u_2=+\delta$ and set $t_{0,1}=t_{0,3}=0$. |
| C | Prescribe $u_2=-\delta$ and set $u_1=u_3=0$. |
| D | Prescribe $u_2=-\delta$ and set $t_{0,1}=t_{0,3}=0$. |

**(b)** Write the total potential $\Pi(\boldsymbol u)$ and explain why there is no prescribed force term from the punch. Complete `forms` using the given energies. Use `ufl.derivative` to compute $R(\boldsymbol u;\boldsymbol v)=-\delta\Pi(\boldsymbol u;\boldsymbol v)$ and its tangent $\delta_u R(\boldsymbol u;\boldsymbol v)[\boldsymbol w]$, where $\boldsymbol v$ is the test function and $\boldsymbol w$ is the displacement increment direction.

See [Chapter 9](https://ceadpx.github.io/finite-element-methods/chapters/11-nonlinear-fem.html) and the [hyperelastic cantilever notebook](https://github.com/CEADpx/finite-element-methods/blob/main/notebooks/nonlinear/hyperelasticity_newton.ipynb) for the energy formulation and UFL differentiation.

In [ ]:
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

# The supplied mesh: an 8 × 4 × 8 mm block with 0.5 mm grid spacing.
domain = mesh.create_box(
    MPI.COMM_SELF, [[0., 0., 0.], [8., 4., 8.]], [16, 8, 16]
)

In [ ]:
def forms(u, v, w, mu, beta, K, dx):
    F = ufl.Identity(3) + ufl.grad(u)
    J = ufl.det(F)
    # Define the energy, residual, and tangent. Return the residual and tangent.
    raise NotImplementedError

## Supplied FE solver

Both tests use the same tetrahedral mesh and continuous piecewise-linear displacement fields. For a given punch width, `make_punch_solver` returns two functions:

- `reset(mu, beta)` sets the parameters and returns the block to zero displacement.
- `attempt(depth)` solves for $\boldsymbol u$ at `depth` in mm, using the previous displacement with the new boundary value imposed as its starting point. It returns `(reason, iterations, force)`. If `reason` is positive, the solve converged and the routine computes `force` from that displacement. Otherwise, `force` is `None`.

In [ ]:
def make_punch_solver(width):
    V = fem.functionspace(domain, ("Lagrange", 1, (3,)))
    u = fem.Function(V)
    v = ufl.TestFunction(V)
    w = ufl.TrialFunction(V)

    # Locate the bottom face and the punch footprint.
    fdim = domain.topology.dim - 1

    def on_punch(x):
        return (np.isclose(x[1], 4.0)
                & (np.abs(x[0] - 4.0) <= width / 2 + 1e-8)
                & (np.abs(x[2] - 4.0) <= width / 2 + 1e-8))

    footprint = mesh.locate_entities_boundary(domain, fdim, on_punch)
    bottom = mesh.locate_entities_boundary(domain, fdim, lambda x: np.isclose(x[1], 0))

    # Select the displacement degrees of freedom for the boundary conditions.
    fixed = fem.locate_dofs_topological(V, fdim, bottom)
    punch_nodes = fem.locate_dofs_topological(V, fdim, footprint)
    normal_dofs = fem.locate_dofs_topological(V.sub(1), fdim, footprint)
    punch_displacement = fem.Constant(domain, PETSc.ScalarType(0.0))
    bcs = [
        fem.dirichletbc(np.zeros(3, dtype=PETSc.ScalarType), fixed, V),
        fem.dirichletbc(punch_displacement, normal_dofs, V.sub(1)),
    ]

    # Check that the marked surface has the area of the square punch.
    tags = mesh.meshtags(domain, fdim, np.sort(footprint), np.ones(len(footprint), dtype=np.int32))
    ds = ufl.Measure("ds", domain=domain, subdomain_data=tags)
    area = fem.assemble_scalar(fem.form(1.0 * ds(1)))
    assert np.isclose(area, width**2)

    mu = fem.Constant(domain, PETSc.ScalarType(0.4))
    beta = fem.Constant(domain, PETSc.ScalarType(0.0))
    dx = ufl.Measure("dx", domain=domain)
    residual, tangent = forms(u, v, w, mu, beta, 2.5, dx)
    # Set up Newton's method with a line search.
    problem = NonlinearProblem(
        residual, u, J=tangent, bcs=bcs, petsc_options_prefix=f"punch_{width}_",
        petsc_options={
            "snes_type": "newtonls", "snes_linesearch_type": "bt",
            "snes_atol": 1e-8, "snes_rtol": 1e-8, "snes_max_it": 40,
            "snes_error_if_not_converged": False,
            "ksp_type": "preonly", "pc_type": "lu",
        },
    )
    raw_residual = fem.form(residual)

    def reset(mu_value, beta_value):
        mu.value = mu_value
        beta.value = beta_value
        punch_displacement.value = 0.0
        u.x.array[:] = 0.0

    def attempt(depth):
        # Update the prescribed displacement and reuse the previous solution.
        punch_displacement.value = -depth
        u.x.array[normal_dofs] = -depth
        problem.solve()
        reason = problem.solver.getConvergedReason()
        iterations = problem.solver.getIterationNumber()
        if reason <= 0:
            return reason, iterations, None

        # Assemble the unmodified residual to recover the vertical punch reaction.
        entries = fem.assemble_vector(raw_residual).array.reshape(-1, 3)
        normal_entries = entries[punch_nodes, 1]
        if normal_entries.min() < -1e-7:
            print(f"Check contact assumption at depth {depth:.2f} mm.")
        return reason, iterations, float(normal_entries.sum())

    return reset, attempt

## 2. Check the load steps (30 marks)

Complete `load_curve` to solve from zero displacement to 1.20 mm in equal steps. Stop if a step fails.

For the **4 mm punch** with $\mu=0.50$ MPa and $\beta=0$, compare four steps of 0.30 mm with eight steps of 0.15 mm. Which runs reach 1.20 mm? If a run fails, give the depth where it fails. Explain why smaller increments can help Newton’s method.

Use **eight equal steps** in Tasks 3 and 4. Steps 2, 4, 6, and 8 give the displacement solutions and forces at the four measured depths.

In [ ]:
reset_narrow, attempt_narrow = make_punch_solver(4)

def load_curve(reset, attempt, mu, beta, n_steps):
    # Return a list with one (depth, reason, iterations, force) tuple per attempted step.
    raise NotImplementedError

# Compare four and eight load steps for the specified neo-Hookean model.

## 3. Select a model using the 4 mm punch data (30 marks)

Compute $E$ for every candidate in the table using the computed and observed forces for the **4 mm punch**. Report the model and parameter values with the smallest $E$, and give that value of $E$.

In [ ]:
measurement_depths = np.array([0.30, 0.60, 0.90, 1.20])
observed_narrow = np.array([4.20, 10.25, 19.90, 34.20])
observed_wide = np.array([6.70, 17.10, 34.90, 63.20])

neo_hookean = [(mu, 0.0) for mu in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8)]
stiffening = [(mu, beta) for mu in (0.3, 0.4, 0.5) for beta in (0.5, 1.0, 1.5)]

In [ ]:
# Find the candidate with the smallest E against observed_narrow.

## 4. Predict the 6 mm punch forces (20 marks)

1. Use the model and parameters selected in Task 3 to predict the forces for the **6 mm punch** without refitting. Plot the predicted and observed forces against depth at the four measured depths.
2. Report the largest absolute relative force error, using the 6 mm data as the reference. Does the prediction meet the **7% tolerance at every depth**?

In [ ]:
reset_wide, attempt_wide = make_punch_solver(6)
# Predict the wider-punch forces using the selected model and parameters.